In [37]:
import sys
!{sys.executable} -m pip install mediapipe


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import cv2
import time
import json


In [39]:
#configuration 

ABSENCE_THRESHOLD = 3

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

print("Face detector loaded successfully")

Face detector loaded successfully


In [ ]:
#Webcam Setup
cap = cv2.VideoCapture("absent_presence.mp4")

if cap.isOpened():
    print("Camera opened successfully")
else:
    print("Camera not detected")

Camera opened successfully


In [41]:

absence_start = None
face_present = True

print("Press 'q' to quit")

while True:
    ret, frame = cap.read()

    if not ret:
        print("\nVideo Finished")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(50, 50)
    )

    current_time = time.time()
     # ----------------------------
    # Face Detected
    # ----------------------------
    if len(faces) > 0:

        face_present = True
        absence_start = None

        # Draw rectangle around face
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y),
                          (x + w, y + h),
                          (0, 255, 0), 2)

        duration_absent = 0

        cv2.putText(
            frame,
            "FACE PRESENT",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    # ----------------------------
    # No Face Detected
    # ----------------------------
    else:

        if absence_start is None:
            absence_start = current_time

        duration_absent = current_time - absence_start

        if duration_absent >= ABSENCE_THRESHOLD:
            face_present = False

        cv2.putText(
            frame,
            f"ABSENT: {duration_absent:.1f}s",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

    # ----------------------------
    # JSON Output
    # ----------------------------
    output = {
    "face_present": face_present,
    "duration_absent_sec": round(duration_absent, 2),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    with open("output.json", "w") as f:
        json.dump(output, f, indent=4)

    # Display Output
    cv2.imshow("Face Presence Detection", frame)

      # Print JSON in terminal
    print(output, end="\r")

    # Quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

        
        

Press 'q' to quit
{'face_present': False, 'duration_absent_sec': 3.22, 'timestamp': '2026-05-13 17:09:09'}
Video Finished
